## Latex-OCR SFT

Here is a demonstration of using python to perform Latex-OCR SFT of Qwen2-VL-2B-Instruct. Through this tutorial, you can quickly understand some details of swift sft, which will be of great help in customizing ms-swift for you~

Are you ready? Let's begin the journey...

In [1]:
# # install ms-swift
!pip install ms-swift qwen-vl-utils -U

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 7.2 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 15.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.3/754.3 kB 21.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 80.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.7/39.7 MB 43.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.6/416.6 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# import some libraries
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

from swift.llm import (
    get_model_tokenizer, load_dataset, get_template, EncodePreprocessor, get_model_arch,
    get_multimodal_target_regex, LazyLLMDataset
)
from swift.utils import get_logger, get_model_parameter_info, plot_images, seed_everything
from swift.tuners import Swift, LoraConfig
from swift.trainers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from functools import partial

logger = get_logger()
seed_everything(42)

2025-07-17 01:04:29.670823: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752714270.039318      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752714270.141615      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[INFO:swift] Successfully registered `/usr/local/lib/python3.11/dist-packages/swift/llm/dataset/data/dataset_info.json`.
[INFO:swift] Global seed set to 42


42

In [3]:
# Hyperparameters for training

system = None  # Using the default system defined in the template.
output_dir = 'output'

# dataset
dataset = ['AI-ModelScope/LaTeX_OCR#20000']  # dataset_id or dataset_path. Sampling 20000 data points
data_seed = 42
max_length = 2048
split_dataset_ratio = 0.01  # Split validation set
num_proc = 4  # The number of processes for data loading.

# lora
lora_rank = 8
lora_alpha = 32
freeze_llm = False
freeze_vit = True
freeze_aligner = True

# training_args
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_checkpointing=True,
    weight_decay=0.1,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    report_to=['tensorboard'],
    logging_first_step=True,
    save_strategy='steps',
    save_steps=50,
    eval_strategy='steps',
    eval_steps=50,
    gradient_accumulation_steps=16,
    # To observe the training results more quickly, this is set to 1 here. 
    # Under normal circumstances, a larger number should be used.
    num_train_epochs=1,
    metric_for_best_model='loss',
    save_total_limit=5,
    logging_steps=5,
    dataloader_num_workers=4,
    data_seed=data_seed,
    remove_unused_columns=False,
)

output_dir = os.path.abspath(os.path.expanduser(output_dir))
logger.info(f'output_dir: {output_dir}')

[INFO:swift] output_dir: /kaggle/working/output


In [4]:
model, processor = get_model_tokenizer(
    model_id_or_path = '/kaggle/input/qwen2-vl/transformers/7b-instruct/1',
    model_type = "qwen2_vl",
    device_map = "auto",
    torch_dtype="bfloat16"
)

[INFO:swift] Loading the model using model_dir: /kaggle/input/qwen2-vl/transformers/7b-instruct/1
[WARNING:swift] Please install the package: `pip install "decord" -U`.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.
[INFO:swift] model_kwargs: {'device_map': 'auto'}


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

[INFO:swift] Setting image_factor: 28. You can adjust this hyperparameter through the environment variable: `IMAGE_FACTOR`.
[INFO:swift] Setting min_pixels: 3136. You can adjust this hyperparameter through the environment variable: `MIN_PIXELS`.
[INFO:swift] Setting max_pixels: 12845056. You can adjust this hyperparameter through the environment variable: `MAX_PIXELS`.
[INFO:swift] Setting max_ratio: 200. You can adjust this hyperparameter through the environment variable: `MAX_RATIO`.
[INFO:swift] Setting video_min_pixels: 100352. You can adjust this hyperparameter through the environment variable: `VIDEO_MIN_PIXELS`.
[INFO:swift] Setting video_max_pixels: 602112. You can adjust this hyperparameter through the environment variable: `VIDEO_MAX_PIXELS`.
[INFO:swift] Setting video_total_pixels: 90316800. You can adjust this hyperparameter through the environment variable: `VIDEO_TOTAL_PIXELS`.
[INFO:swift] Setting frame_factor: 2. You can adjust this hyperparameter through the environmen

In [5]:
# Obtain the model and template

template = get_template(model.model_meta.template, processor, default_system=system, max_length=max_length)
template.set_mode('train')
if template.use_model:
    template.model = model

# Get target_modules and add trainable LoRA modules to the model.
target_modules = get_multimodal_target_regex(model, freeze_llm=freeze_llm, freeze_vit=freeze_vit, 
                            freeze_aligner=freeze_aligner)
lora_config = LoraConfig(task_type='CAUSAL_LM', r=lora_rank, lora_alpha=lora_alpha,
                         target_modules=target_modules)
model = Swift.prepare_model(model, lora_config)
logger.info(f'lora_config: {lora_config}')

# Print model structure and trainable parameters.
logger.info(f'model: {model}')
model_parameter_info = get_model_parameter_info(model)
logger.info(f'model_parameter_info: {model_parameter_info}')

[INFO:swift] default_system: 'You are a helpful assistant.'
[INFO:swift] max_length: 2048
[INFO:swift] response_prefix: ''
[INFO:swift] agent_template: hermes
[INFO:swift] norm_bbox: norm1000
[INFO:swift] lora_config: LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='/kaggle/input/qwen2-vl/transformers/7b-instruct/1', revision=None, inference_mode=False, r=8, target_modules='^(model.language_model.*\\.(q_proj|up_proj|down_proj|k_proj|v_proj|o_proj|gate_proj))$', exclude_modules=None, lora_alpha=32, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=Fal

In [6]:
# Download and load the dataset, split it into a training set and a validation set,
# and encode the text data into tokens.
train_dataset, val_dataset = load_dataset(dataset, split_dataset_ratio=split_dataset_ratio, num_proc=num_proc,
                                          seed=data_seed)

logger.info(f'train_dataset: {train_dataset}')
logger.info(f'val_dataset: {val_dataset}')
logger.info(f'train_dataset[0]: {train_dataset[0]}')

train_dataset = LazyLLMDataset(train_dataset, template.encode, random_state=data_seed)
val_dataset = LazyLLMDataset(val_dataset, template.encode, random_state=data_seed)
data = train_dataset[0]
logger.info(f'encoded_train_dataset[0]: {data}')

template.print_inputs(data)

[INFO:swift] Downloading the dataset from ModelScope, dataset_id: AI-ModelScope/LaTeX_OCR


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map (num_proc=4):   0%|          | 0/76318 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/8475 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/9443 [00:00<?, ? examples/s]

[INFO:swift] train_dataset: Dataset({
    features: ['images', 'messages'],
    num_rows: 19800
})
[INFO:swift] val_dataset: Dataset({
    features: ['images', 'messages'],
    num_rows: 200
})
[INFO:swift] train_dataset[0]: {'images': [{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01\xf4\x00\x00\x00d\x08\x02\x00\x00\x00\xff\xa5U*\x00\x00%\xc9IDATx\x9c\xed\x9dyX\x14G\xfa\xc7\xab{\xee\x81A\x18D1QLL\x8c\x1a\x9f\xa8H@E<Q\xc1\xa8x\x10\xaf\x18\x8dD\x9f\xec\x121 \x1e`@T\x8c\x8a"b\xe2\x19\xaf\xe4\xd9\xa8`\xbc@QQ\xf1@Qte\xdd\xb8\xd9$\x1e\xd1\x185^0\x80sO\xf7t\xd7\xef\x8fw\xe9_\x87[\x19\xa2\x8e\xf5\xf9\xc3gl\xba\xab\xab\xaa\xdf\xfe\xd6\xdbo]\x14\xc6\x18\x11\x08\x04\x02\xc1\xb9\x90>\xeb\x0c\x10\x08\x84\x97\x05\xc1\x95\xa4(\xea\xd9\xe6\xe4e\x80~\xd6\x19 \x10\x08/\x0bT\x05<\xcf?\xeb\xbc8?\xc4s\x7f20\xc6v\xbb\x1d!$\x95J\x89\xf7A T\x05c\xcc\xf3<M\xd3\xc2\x0b\x821\x86\xdfw\xee\xdc\xa1(J\xab\xd5\xaa\xd5\xeag\x9a\xc7\x97\x02\x8a\xc4\xdc\xeb\x8f`\xa3\x04\x02\xa1\x9e\xb0,K\xd34\xc7qc\xc6\x8c\x

In [ ]:
# Get the trainer and start the training.
model.enable_input_require_grads()  # Compatible with gradient checkpointing
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=template.data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    template=template,
)
trainer.train()

last_model_checkpoint = trainer.state.last_model_checkpoint
logger.info(f'last_model_checkpoint: {last_model_checkpoint}')

/usr/local/lib/python3.11/dist-packages/swift/trainers/mixin.py:95: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
[INFO:swift] use_reentrant: True
[INFO:swift] Successfully registered post_encode hook: ['PeftModelForCausalLM'].
Train:   0%|          | 0/1238 [00:00<?, ?it/s][INFO:swift] use_logits_to_keep: False
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
Train:   0%|          | 1/1238 [01:03<21:48:39, 63.48s/it]

{'loss': 0.31032991, 'token_acc': 0.9286859, 'grad_norm': 0.79776788, 'learning_rate': 1.61e-06, 'memory(GiB)': 17.32, 'train_speed(iter/s)': 0.015622, 'epoch': 0.0, 'global_step/max_steps': '1/1238', 'percentage': '0.08%', 'elapsed_time': '1m 3s', 'remaining_time': '21h 48m 44s'}


Train:   0%|          | 5/1238 [04:21<17:11:49, 50.21s/it]

{'loss': 0.40863571, 'token_acc': 0.91045499, 'grad_norm': 0.83251452, 'learning_rate': 8.06e-06, 'memory(GiB)': 17.91, 'train_speed(iter/s)': 0.019079, 'epoch': 0.0, 'global_step/max_steps': '5/1238', 'percentage': '0.40%', 'elapsed_time': '4m 21s', 'remaining_time': '17h 54m 53s'}


Train:   1%|          | 10/1238 [08:38<17:39:40, 51.78s/it]

...

Train:   2%|▏         | 20/1238 [17:32<17:45:29, 52.49s/it]

{'loss': 0.19042133, 'token_acc': 0.94452031, 'grad_norm': 0.54309493, 'learning_rate': 3.226e-05, 'memory(GiB)': 17.92, 'train_speed(iter/s)': 0.018996, 'epoch': 0.02, 'global_step/max_steps': '20/1238', 'percentage': '1.62%', 'elapsed_time': '17m 32s', 'remaining_time': '17h 48m 5s'}


Train:   2%|▏         | 25/1238 [21:54<17:37:33, 52.31s/it]

{'loss': 0.2107111, 'token_acc': 0.94177511, 'grad_norm': 0.48334485, 'learning_rate': 4.032e-05, 'memory(GiB)': 17.92, 'train_speed(iter/s)': 0.019011, 'epoch': 0.02, 'global_step/max_steps': '25/1238', 'percentage': '2.02%', 'elapsed_time': '21m 54s', 'remaining_time': '17h 42m 58s'}


Train:   2%|▏         | 29/1238 [25:21<17:20:38, 51.64s/it]

In [ ]:
# Visualize the training loss.
# You can also use the TensorBoard visualization interface during training by entering
# `tensorboard --logdir '{output_dir}/runs'` at the command line.
images_dir = os.path.join(output_dir, 'images')
logger.info(f'images_dir: {images_dir}')
plot_images(images_dir, training_args.logging_dir, ['train/loss'], 0.9)  # save images

# Read and display the image.
# The light yellow line represents the actual loss value,
# while the yellow line represents the loss value smoothed with a smoothing factor of 0.9.
from IPython.display import display
from PIL import Image
image = Image.open(os.path.join(images_dir, 'train_loss.png'))
display(image)

# 推理

In [ ]:
# import some libraries
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from swift.llm import (
    InferEngine, InferRequest, PtEngine, RequestConfig, get_template, load_dataset, load_image
)
from swift.utils import get_model_parameter_info, get_logger, seed_everything
logger = get_logger()
seed_everything(42)

In [ ]:
# Hyperparameters for inference
last_model_checkpoint = 'output/checkpoint-xxx'

# model
model_id_or_path = 'Qwen/Qwen2-VL-2B-Instruct'  # model_id or model_path
system = None
infer_backend = 'pt'

# dataset
dataset = ['AI-ModelScope/LaTeX_OCR#20000']
data_seed = 42
split_dataset_ratio = 0.01
num_proc = 4
strict = False

# generation_config
max_new_tokens = 512
temperature = 0
stream = True

In [ ]:
# Get model and template, and load LoRA weights.
engine = PtEngine(model_id_or_path, adapters=[last_model_checkpoint])
template = get_template(engine.model_meta.template, engine.processor, default_system=system)
# The default mode of the template is 'pt', so there is no need to make any changes.
# template.set_mode('pt')

model_parameter_info = get_model_parameter_info(engine.model)
logger.info(f'model_parameter_info: {model_parameter_info}')

In [ ]:
# Due to the data_seed setting, the validation set here is the same as the validation set used during training.
_, val_dataset = load_dataset(dataset, split_dataset_ratio=split_dataset_ratio, num_proc=num_proc,
                              strict=strict, seed=data_seed)
val_dataset = val_dataset.select(range(10))  # Take the first 10 items

In [ ]:
# Streaming inference and save images from the validation set.
# The batch processing code can be found here: https://github.com/modelscope/ms-swift/blob/main/examples/infer/demo_mllm.py
def infer_stream(engine: InferEngine, infer_request: InferRequest):
    request_config = RequestConfig(max_tokens=max_new_tokens, temperature=temperature, stream=True)
    gen_list = engine.infer([infer_request], request_config)
    query = infer_request.messages[0]['content']
    print(f'query: {query}\nresponse: ', end='')
    for resp in gen_list[0]:
        if resp is None:
            continue
        print(resp.choices[0].delta.content, end='', flush=True)
    print()

from IPython.display import display
os.makedirs('images', exist_ok=True)
for i, data in enumerate(val_dataset):
    image = data['images'][0]
    image = load_image(image['bytes'] or image['path'])
    image.save(f'images/{i}.png')
    display(image)
    infer_stream(engine, InferRequest(**data))
    print('-' * 50)